In [ ]:
from config import P, L
from math_utils import random_a
from cycle_utils import gen_cycles, count_cycles
from matrix_builder import gen_h_xz, gen_c_constraints
from solution_space import extract_b_solution_space, count_total_solutions, find_b_with_numpy, get_snf_basis, find_b_incremental_numpy
from z3_solver import find_b_with_z3, find_b_with_z3_staged

In [ ]:
import time
import math
import random
from config import P, L
from cycle_utils import gen_cycles
from matrix_builder import gen_h_xz

def solve_linear_congruence(C, D, mod_val):
    C = C % mod_val
    D = D % mod_val
    g = math.gcd(C, mod_val)
    
    if D % g != 0:
        return []
        
    if g == mod_val:
        return list(range(mod_val))
        
    C_prime = C // g
    D_prime = D // g
    P_prime = mod_val // g
    
    inv_C = pow(C_prime, -1, P_prime)
    base_x = (D_prime * inv_C) % P_prime
    
    return [(base_x + k * P_prime) % mod_val for k in range(g)]

def fast_dfs_solve(cycles_4, cycles_6, h_x, h_z, p_val=P):
    COPRIMES = [a for a in range(1, p_val) if math.gcd(a, p_val) == 1]
    inv_memo = {a: pow(a, -1, p_val) for a in COPRIMES}

    # 各回の探索にランダム性を持たせるためシャッフル
    random.shuffle(COPRIMES)

    # 制約の連鎖が最も早く閉じる順番で変数を割り当てる
    assignment_order = [0, 6, 1, 7, 2, 8, 3, 9, 4, 10, 5, 11]
    step_of_idx = {idx: step for step, idx in enumerate(assignment_order)}
    
    checks_at_step = [[] for _ in range(12)]

    # 1. 条件A・条件Bのチェック登録
    for i in range(6):
        for j in range(6):
            idx1 = i
            idx2 = 6 + j
            step = max(step_of_idx[idx1], step_of_idx[idx2])
            is_non_commutative = (6*i + j) in [3, 8]
            checks_at_step[step].append(('AB', i, 6+j, is_non_commutative))

    # 2. 条件C (サイクル) のチェック登録: XとZを区別する
    for cycle in cycles_4 + cycles_6:
        idx_x = [h_x[r][c] for r, c in cycle]
        step_x = max(step_of_idx[idx] for idx in idx_x)
        checks_at_step[step_x].append(('C_X', idx_x))

        idx_z = [h_z[r][c] for r, c in cycle]
        step_z = max(step_of_idx[idx] for idx in idx_z)
        checks_at_step[step_z].append(('C_Z', idx_z))

    a_vec = [0] * 12
    b_vec = [0] * 12
    node_count = 0
    start_time = time.time()

    def get_b_candidates(step):
        if step == 0: 
            cands = list(range(p_val))
            random.shuffle(cands)
            return cands
            
        elif step == 1:  C, D = (a_vec[0] - 1), ((a_vec[6] - 1) * b_vec[0])
        elif step == 2:  C, D = (1 - a_vec[6]), ((1 - a_vec[1]) * b_vec[6])
        elif step == 3:  C, D = (a_vec[0] - 1), ((a_vec[7] - 1) * b_vec[0])
        elif step == 4:  C, D = (1 - a_vec[6]), ((1 - a_vec[2]) * b_vec[6])
        elif step == 5:  C, D = (a_vec[0] - 1), ((a_vec[8] - 1) * b_vec[0])
        elif step == 6:  C, D = (1 - a_vec[6]), ((1 - a_vec[3]) * b_vec[6])
        elif step == 7:  C, D = (a_vec[1] - 1), ((a_vec[9] - 1) * b_vec[1]) 
        elif step == 8:  C, D = (1 - a_vec[6]), ((1 - a_vec[4]) * b_vec[6])
        elif step == 9:  C, D = (a_vec[0] - 1), ((a_vec[10] - 1) * b_vec[0])
        elif step == 10: C, D = (1 - a_vec[6]), ((1 - a_vec[5]) * b_vec[6])
        elif step == 11: C, D = (a_vec[0] - 1), ((a_vec[11] - 1) * b_vec[0])

        return solve_linear_congruence(C, D, p_val)

    def solve(step):
        nonlocal node_count
        node_count += 1
        
        if node_count % 50000 == 0:
            print(f"\r... 探索中: {node_count} ノードを検証 (経過時間: {time.time() - start_time:.2f}秒)", end="", flush=True)

        if step == 12:
            return True 

        current_idx = assignment_order[step]
        checks = checks_at_step[step]

        current_coprimes = random.sample(COPRIMES, len(COPRIMES))

        for a_val in current_coprimes:
            a_vec[current_idx] = a_val
            b_candidates = get_b_candidates(step)
            
            for b_val in b_candidates:
                b_vec[current_idx] = b_val

                valid = True
                for check in checks:
                    ctype = check[0]
                    if ctype == 'AB':
                        _, i, j, is_non_comm = check
                        val = ((1 - a_vec[j]) * b_vec[i] + (a_vec[i] - 1) * b_vec[j]) % p_val
                        if is_non_comm:
                            if val == 0: valid = False; break
                        else:
                            if val != 0: valid = False; break
                            
                    elif ctype == 'C_X':
                        idx_list = check[1]
                        A, B = 1, 0
                        for k in range(len(idx_list) // 2):
                            # X側: 順方向 -> 逆方向
                            af, bf = a_vec[idx_list[2*k]], b_vec[idx_list[2*k]]
                            ab, bb = a_vec[idx_list[2*k+1]], b_vec[idx_list[2*k+1]]
                            
                            # 順方向
                            A = (af * A) % p_val
                            B = (af * B + bf) % p_val
                            
                            # 逆方向
                            inv_a = inv_memo[ab]
                            A = (inv_a * A) % p_val
                            B = (inv_a * (B - bb + p_val)) % p_val

                        g = math.gcd((A - 1) % p_val, p_val)
                        if B % g == 0:
                            valid = False
                            break
                            
                    elif ctype == 'C_Z':
                        idx_list = check[1]
                        A, B = 1, 0
                        for k in range(len(idx_list) // 2):
                            # Z側: 逆方向 -> 順方向
                            ab, bb = a_vec[idx_list[2*k]], b_vec[idx_list[2*k]]
                            af, bf = a_vec[idx_list[2*k+1]], b_vec[idx_list[2*k+1]]
                            
                            # 逆方向
                            inv_a = inv_memo[ab]
                            A = (inv_a * A) % p_val
                            B = (inv_a * (B - bb + p_val)) % p_val
                            
                            # 順方向
                            A = (af * A) % p_val
                            B = (af * B + bf) % p_val

                        g = math.gcd((A - 1) % p_val, p_val)
                        if B % g == 0:
                            valid = False
                            break

                if valid:
                    if solve(step + 1):
                        return True

        return False 

    print("\n深さ優先探索(DFS)による超高速逐次探索を開始する...")
    
    if solve(0):
        print(f"\n【大成功！】 計算時間: {time.time() - start_time:.3f}秒 (検証ノード数: {node_count})")
        return a_vec, b_vec
    else:
        print(f"\n解空間を全て探索したが、解が存在しなかった。({time.time() - start_time:.3f}秒)")
        return None, None
# ==================================================
# 実行部分 (main.py 等に配置)
# ==================================================
if __name__ == "__main__":
    N = 10
    results = []
    cycles_4 = gen_cycles([4])
    cycles_6 = gen_cycles([6])
    h_x, h_z = gen_h_xz()
    
    for i in range(N):
        print(f"\n=== {i + 1}個目の解を探索 ===")
        a_sol, b_sol = fast_dfs_solve(cycles_4, cycles_6, h_x, h_z, P)
        
        if a_sol and b_sol:
            print(f"a_vec = {a_sol}")
            print(f"b_vec = {b_sol}")
            results.append((a_sol, b_sol))
            
    print(f"\n探索完了。{len(results)}個の解を取得した。")

In [ ]:
import z3
import time
import math
from config import P, L
from cycle_utils import gen_cycles
from matrix_builder import gen_h_xz

def find_ab_simultaneously(cycles_4, cycles_6, h_x, h_z, p_val=P):
    """
    逆数演算を排除した数理モデルを用いて、Z3に a と b を同時に探索させる究極の関数
    """
    solver = z3.Solver()
    P_bv = z3.BitVecVal(p_val, 32)
    
    # 【最適化】定数ノードを事前生成して、Z3のメモリ消費と構築時間を大幅に削減
    X_BVS = [z3.BitVecVal(x, 32) for x in range(p_val)]
    
    # 互いに素な a の候補リスト
    coprimes = [i for i in range(1, p_val) if math.gcd(i, p_val) == 1]
    
    # 探索変数: a_0 ~ a_11, b_0 ~ b_11
    a_vars = [z3.BitVec(f'a_{i}', 32) for i in range(L)]
    b_vars = [z3.BitVec(f'b_{i}', 32) for i in range(L)]
    
    # ドメイン制約
    for a in a_vars:
        solver.add(z3.Or([a == z3.BitVecVal(c, 32) for c in coprimes]))
    for b in b_vars:
        solver.add(z3.UGE(b, 0), z3.ULT(b, P_bv))
        
    # ==================================================
    # 条件A(可換) と 条件B(非可換) の制約
    # ==================================================
    for i in range(6):
        for j in range(6):
            idx = 6*i + j
            left = a_vars[6+j] * b_vars[i] + b_vars[6+j]
            right = b_vars[i] + a_vars[i] * b_vars[6+j]
            
            if idx in [3, 8]:
                solver.add(z3.URem(left, P_bv) != z3.URem(right, P_bv)) # 条件B
            else:
                solver.add(z3.URem(left, P_bv) == z3.URem(right, P_bv)) # 条件A
                
    # ==================================================
    # 条件C (ガース制約) : 逆数を使わない (A, B, C) 追跡モデル
    # ==================================================
    def get_abc_expr_x(cycle):
        idx_list = [h_x[r][c] for r, c in cycle]
        A, B, C = z3.BitVecVal(1, 32), z3.BitVecVal(0, 32), z3.BitVecVal(1, 32)
        
        for i in range(len(cycle) // 2):
            # 順方向: f(x) = a x + b
            a_f, b_f = a_vars[idx_list[2*i]], b_vars[idx_list[2*i]]
            A_new = z3.URem(a_f * A, P_bv)
            B_new = z3.URem(a_f * B + b_f * C, P_bv)
            C_new = C
            A, B, C = A_new, B_new, C_new
            
            # 逆方向: f^-1(x) = (x - b) / a
            a_inv, b_inv = a_vars[idx_list[2*i+1]], b_vars[idx_list[2*i+1]]
            A_new = A
            B_new = z3.URem(B + P_bv - z3.URem(b_inv * C, P_bv), P_bv)
            C_new = z3.URem(a_inv * C, P_bv)
            A, B, C = A_new, B_new, C_new
            
        return A, B, C
    
    def get_abc_expr_z(cycle):
        idx_list = [h_z[r][c] for r, c in cycle]
        A, B, C = z3.BitVecVal(1, 32), z3.BitVecVal(0, 32), z3.BitVecVal(1, 32)
        
        for i in range(len(cycle) // 2):
            # 逆方向: f^-1(x) = (x - b) / a
            a_inv, b_inv = a_vars[idx_list[2*i]], b_vars[idx_list[2*i]]
            A_new = A
            B_new = z3.URem(B + P_bv - z3.URem(b_inv * C, P_bv), P_bv)
            C_new = z3.URem(a_inv * C, P_bv)
            A, B, C = A_new, B_new, C_new
            
            # 順方向: f(x) = a x + b
            a_f, b_f = a_vars[idx_list[2*i+1]], b_vars[idx_list[2*i+1]]
            A_new = z3.URem(a_f * A, P_bv)
            B_new = z3.URem(a_f * B + b_f * C, P_bv)
            C_new = C
            A, B, C = A_new, B_new, C_new
            
        return A, B, C 

    def add_cycles_to_solver(cycle_list):
        """指定されたサイクルの制約をソルバに追加するヘルパー関数"""
        for cycle in cycle_list:
            A_x, B_x, C_x = get_abc_expr_x(cycle)
            A_z, B_z, C_z = get_abc_expr_z(cycle)
            
            for x_val in range(p_val):
                # 事前生成した定数ノードを使い回す
                x_bv = X_BVS[x_val]
                
                left_x = z3.URem(A_x * x_bv + B_x, P_bv)
                right_x = z3.URem(C_x * x_bv, P_bv)
                solver.add(left_x != right_x)
                
                left_z = z3.URem(A_z * x_bv + B_z, P_bv)
                right_z = z3.URem(C_z * x_bv, P_bv)
                solver.add(left_z != right_z)

    # ==================================================
    # 【最適化】段階的探索（インクリメンタル・ソルビング）
    # ==================================================
    print("Z3の制約式を構築中 (長さ4のサイクル)...")
    start_build = time.time()
    add_cycles_to_solver(cycles_4)
    print(f"構築完了 ({time.time() - start_build:.2f}秒)")

    print("第1段階: 条件A, B, C(長さ4) の探索を開始します...")
    start_time = time.time()
    res = solver.check()
    
    if res != z3.sat:
        print(f"長さ4を回避できる a, b は存在しませんでした (実行時間: {time.time() - start_time:.2f}秒)")
        return None, None
        
    print(f"長さ4をクリア！ 引き続き長さ6の制約式を構築します...")
    start_build = time.time()
    add_cycles_to_solver(cycles_6)
    print(f"構築完了 ({time.time() - start_build:.2f}秒)")
    
    print("第2段階: 長さ6を含めた完全探索を開始します...")
    res = solver.check()
    elapsed = time.time() - start_time
    
    if res == z3.sat:
        print(f"\n【大成功！】計算時間: {elapsed:.2f}秒")
        model = solver.model()
        ans_a = [model[a].as_long() for a in a_vars]
        ans_b = [model[b].as_long() for b in b_vars]
        return ans_a, ans_b
    else:
        print(f"\n解が見つかりませんでした (実行時間: {elapsed:.2f}秒 / 結果: {res})")
        return None, None

# ==================================================
# メイン実行スクリプト
# ==================================================
if __name__ == "__main__":
    # 事前準備
    cycles_4 = gen_cycles([4])
    cycles_6 = gen_cycles([6])
    h_x, h_z = gen_h_xz()
    
    # Z3による a, b 同時探索を実行
    a_sol, b_sol = find_ab_simultaneously(cycles_4, cycles_6, h_x, h_z, P)
    
    if a_sol and b_sol:
        print(f"a_vec = {a_sol}")
        print(f"b_vec = {b_sol}")

In [ ]:
import time
import math

N = 10
attempts = []
results = []
cycles_4 = gen_cycles([4])
cycles_6 = gen_cycles([6])
h_x, h_z = gen_h_xz()

for i in range(N):
    fail_count = 0
    print(f"\n=== {i + 1}個目の探索を開始 ===")
    
    while True:
        print(f"\r[{fail_count}] a_vec生成...", end="", flush=True)
        a_vec = [random_a(P) for _ in range(L)]
        
        print(" SNF抽出...", end="", flush=True)
        ker_Ga_s, Gb = get_snf_basis(a_vec, P)
        
        if not ker_Ga_s:
            print(" 完了 (解空間なし)", flush=True)
            fail_count += 1
            continue
            
        # 探索する総組み合わせ数を計算
        total_space = math.prod([info['num_vals'] for info in ker_Ga_s])
            
        print(" 制約生成...", end="", flush=True)
        constraints_4 = gen_c_constraints(cycles_4, a_vec, h_x, h_z)
        constraints_6 = gen_c_constraints(cycles_6, a_vec, h_x, h_z)
        
        print(f" NumPy段階的全探索(総数:{total_space})...", end="", flush=True)
        search_start_time = time.time()
        
        # NumPyによる段階的・全探索を実行
        b_sol = find_b_incremental_numpy(ker_Ga_s, Gb, constraints_4, constraints_6, P)
        
        search_elapsed = time.time() - search_start_time

        if b_sol is not None:
            print(f" 【大成功！】計算時間: {search_elapsed:.3f}秒", flush=True)
            print(f"すべての条件(A, B, C)を満たす b_vec: {list(b_sol)}")
            results.append([a_vec, list(b_sol)])
            attempts.append(fail_count + 1)
            break
        else:
            print(f" 失敗(条件B,C4,C6で全滅) 時間: {search_elapsed:.3f}秒", flush=True)
            fail_count += 1
            
print(f"\n探索完了。{N}個の解を results に格納しました。")

In [ ]:
import time

N = 10
attempts = []
results = []
cycles_4 = gen_cycles([4])
cycles_6 = gen_cycles([6])
h_x, h_z = gen_h_xz()

for i in range(N):
    fail_count = 0
    print(f"\n=== {i + 1}個目の探索を開始 ===")
    
    while True:
        # 進捗を1行で上書きしながらリアルタイム表示
        print(f"\r[{fail_count}] a_vec生成...", end="", flush=True)
        a_vec = [random_a(P) for _ in range(L)]
        
        print(" SNF抽出...", end="", flush=True)
        # 組み合わせ爆発を防ぐため constraints_4=None でベースの解空間のみ抽出
        free, const, valid_pats = extract_b_solution_space(a_vec, constraints_4=None, p_val=P)
        total = count_total_solutions(free, valid_pats)
        
        if total == 0:
            print(" 完了 (解空間なし)", flush=True)
            fail_count += 1
            continue
            
        print(" 制約生成...", end="", flush=True)
        constraints_4 = gen_c_constraints(cycles_4, a_vec, h_x, h_z)
        constraints_6 = gen_c_constraints(cycles_6, a_vec, h_x, h_z)
        
        # NumPyで一括検査するため、長さ4と長さ6の制約行列を結合する
        constraints_all = constraints_4 + constraints_6
        
        print(f" NumPy一括探索(総数:{total})...", end="", flush=True)
        search_start_time = time.time()
        
        # NumPyによる高速一括探索
        b_sol = find_b_with_numpy(free, const, valid_pats, constraints_all, P)
        
        search_elapsed = time.time() - search_start_time

        if b_sol is not None:
            print(f" 【大成功！】計算時間: {search_elapsed:.3f}秒", flush=True)
            print(f"すべての条件(A, B, C)を満たす b_vec: {list(b_sol)}")
            results.append([a_vec, list(b_sol)])
            attempts.append(fail_count + 1)
            break
        else:
            print(f" 失敗(長さ4または6で全滅) 時間: {search_elapsed:.3f}秒", flush=True)
            fail_count += 1
        
print(f"\n探索完了。{N}個の解を results に格納しました。")

In [ ]:
# import time

# N = 10
# attempts = []
# results = []
# cycles_4 = gen_cycles([4])
# cycles_6 = gen_cycles([6])
# h_x, h_z = gen_h_xz()

# for i in range(N):
#     fail_count = 0
#     print(f"\n=== {i + 1}個目の探索を開始 ===")
    
#     while True:
#         # 進捗を1行で上書きしながらリアルタイム表示
#         print(f"\r[{fail_count}] a_vec生成...", end="", flush=True)
#         a_vec = [random_a(P) for _ in range(L)]
        
#         print(" SNF抽出...", end="", flush=True)
#         free, const, valid_pats = extract_b_solution_space(a_vec, constraints_4=None, p_val=P)
#         total = count_total_solutions(free, valid_pats)
        
#         if total == 0:
#             print(" 完了 (解空間なし)", flush=True)
#             fail_count += 1
#             continue
            
#         print(" 制約生成...", end="", flush=True)
#         constraints_4 = gen_c_constraints(cycles_4, a_vec, h_x, h_z)
#         constraints_6 = gen_c_constraints(cycles_6, a_vec, h_x, h_z)
        
#         print(f" Z3探索(総数:{total})...", end="", flush=True)
#         z3_start_time = time.time()
        
#         # Z3実行 (タイムアウトが設定されているので2秒以内に必ず返ってくるはず)
#         passed_stage1, b_sol = find_b_with_z3_staged(free, const, valid_pats, constraints_4, constraints_6, P)
#         z3_elapsed = time.time() - z3_start_time

#         if b_sol is not None:
#             print(f" 【成功】Z3実行時間: {z3_elapsed:.2f}秒", flush=True)
#             print(f"すべての条件(A, B, C)を満たす b_vec: {b_sol}")
#             results.append([a_vec, b_sol])
#             attempts.append(fail_count + 1)
#             break

#         else:
#             if passed_stage1:
#                 print(f" 失敗(長さ6で詰まり) 時間: {z3_elapsed:.2f}秒", flush=True)
#             else:
#                 if z3_elapsed > 1.5:
#                     print(f" 失敗(Z3タイムアウト) 時間: {z3_elapsed:.2f}秒", flush=True)
#                 else:
#                     print(f" 失敗(即死判定) 時間: {z3_elapsed:.2f}秒", flush=True)
            
#             fail_count += 1

# print(f"\n探索完了。{N}個の解を results に格納しました。")

In [ ]:
for a, b in results:
    print(count_cycles(a, b))

In [ ]:
# cycles = gen_cycles(6)
# h_x, h_z = gen_h_xz()
# for a_vec, _ in results:
#     res = find_b_from_a(a_vec, cycles, h_x, h_z)
#     print(res)